# Training

#### Feature filtering helper
Define a helper to drop tools with unknown training sets and optionally exclude clinical-trained or VEP-derived features depending on the model version.

In [ ]:
def filter_features(df, model_name, remove_clinical_trained=False, remove_all_veps=False):
    with open("../resources/feature_lists/veps_excluded_due_to_unavailable_training_sets.txt", "r") as f:
        no_training_set = [line.strip() for line in f]
    df = df.drop(columns=no_training_set, errors="ignore")
    
    if model_name in ["FuncVEP_CTI", "ClinVEP_CTI"]:
        return df
    
    if model_name == "FuncVEP_CTE" or model_name == "ClinVEP_CTE" or remove_clinical_trained:
        with open("../resources/feature_lists/clinical_trained_veps.txt", "r") as f:
            clinical_trained_veps = [line.strip() for line in f]
        df = df.drop(columns=clinical_trained_veps, errors="ignore")
    
    if model_name == "FuncVEP_SP" or model_name == "ClinVEP_SP" or remove_all_veps:
        with open("../resources/feature_lists/all_veps.txt", "r") as f:
            veps = [line.strip() for line in f]
        df = df.drop(columns=veps, errors="ignore")

    return df

## FuncVEP training function
LightGBM + Optuna training on functional labels (PS3 vs. BS3/proxy_benign), saving the best model, feature list, and training variant IDs.

In [ ]:
def train_funcvep(
    df,
    model_name,
    test_size=0.1,
    n_trials=50,
    random_state=42,
    remove_clinical_trained=False,
    remove_all_veps=False,
):
    import os
    import json
    import warnings
    import joblib
    import optuna
    import pandas as pd
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, accuracy_score

    warnings.simplefilter("ignore")

    target_column = "functional_training_label"
    id_column = "ID"
    ensg_column = "ensg"

    model_dir = f"../models/{model_name}"
    os.makedirs(model_dir, exist_ok=True)

    df = filter_features(df, model_name, remove_clinical_trained, remove_all_veps)
    df = df.dropna(subset=[target_column])

    required_cols = {id_column, ensg_column, target_column}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Map labels to 0/1
    df[target_column] = df[target_column].replace({"PS3": 1, "BS3": 0, "proxy_benign": 0})
    df = df.dropna(subset=[target_column])

    non_feature_cols = {id_column, ensg_column, target_column}
    feature_columns = [c for c in df.columns if c not in non_feature_cols]

    pd.DataFrame(feature_columns, columns=["Feature"]).to_csv(
        os.path.join(model_dir, "training_features.txt"),
        sep="\t",
        index=False,
    )

    df[feature_columns] = df[feature_columns].apply(pd.to_numeric, errors="coerce")

    # Save all training variants (train + internal eval) to avoid generating scores for them later on
    training_variants_df = df[[id_column, ensg_column]].drop_duplicates()
    training_variants_df.to_csv(
        os.path.join(model_dir, "training_set.txt"),
        sep="\t",
        index=False,
    )

    X = df[feature_columns]
    y = df[target_column].astype(int)
    valid_mask = X.notna().any(axis=1)
    X = X[valid_mask]
    y = y[valid_mask]

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
    )

    def objective(trial):
        params = {
            "num_leaves": trial.suggest_int("num_leaves", 16, 128),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "n_estimators": trial.suggest_int("n_estimators", 100, 600),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
            "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-4, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-4, 10.0, log=True),
            "objective": "binary",
            "random_state": random_state,
            "verbose": -1,
        }

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="auc",
            callbacks=[lgb.log_evaluation(0)],
        )

        y_pred_proba = model.predict_proba(X_val)[:, 1]
        return roc_auc_score(y_val, y_pred_proba)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    best_params = study.best_params
    best_params["objective"] = "binary"
    best_params["random_state"] = random_state

    lgb_model = lgb.LGBMClassifier(**best_params)
    lgb_model.fit(X_train, y_train)

    y_val_proba = lgb_model.predict_proba(X_val)[:, 1]
    y_val_pred = lgb_model.predict(X_val)

    auc_score = roc_auc_score(y_val, y_val_proba)
    accuracy = accuracy_score(y_val, y_val_pred)

    print(f"Validation AUC: {auc_score:.6f}")
    print(f"Validation accuracy: {accuracy * 100:.2f}%")

    joblib.dump(lgb_model, os.path.join(model_dir, "model.pkl"))

    with open(os.path.join(model_dir, "best_params.json"), "w") as f:
        json.dump(best_params, f, indent=2)


## ClinVEP training function
LightGBM + Optuna training on balanced ClinVar labels (P vs. B), saving the best model, feature list, and training variant IDs.

In [ ]:
def train_clinvep(
    df,
    model_name,
    test_size=0.1,
    n_trials=50,
    random_state=42,
    remove_clinical_trained=False,
    remove_all_veps=False,
):
    import os
    import json
    import warnings
    import joblib
    import optuna
    import pandas as pd
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, accuracy_score

    warnings.simplefilter("ignore")

    target_column = "clinical_training_label"
    id_column = "ID"
    ensg_column = "ensg"

    model_dir = f"../models/{model_name}"
    os.makedirs(model_dir, exist_ok=True)

    df = filter_features(df, model_name, remove_clinical_trained, remove_all_veps)
    df = df.dropna(subset=[target_column])

    required_cols = {id_column, ensg_column, target_column}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df[target_column] = df[target_column].replace({"P": 1, "B": 0})
    df = df.dropna(subset=[target_column])

    non_feature_cols = {id_column, ensg_column, target_column}
    feature_columns = [c for c in df.columns if c not in non_feature_cols]

    pd.DataFrame(feature_columns, columns=["Feature"]).to_csv(
        os.path.join(model_dir, "training_features.txt"),
        sep="\t",
        index=False,
    )

    df[feature_columns] = df[feature_columns].apply(pd.to_numeric, errors="coerce")

    training_variants_df = df[[id_column, ensg_column]].drop_duplicates()
    training_variants_df.to_csv(
        os.path.join(model_dir, "training_set.txt"),
        sep="\t",
        index=False,
    )

    X = df[feature_columns]
    y = df[target_column].astype(int)
    valid_mask = X.notna().any(axis=1)
    X = X[valid_mask]
    y = y[valid_mask]

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
    )

    def objective(trial):
        params = {
            "num_leaves": trial.suggest_int("num_leaves", 16, 128),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "n_estimators": trial.suggest_int("n_estimators", 100, 600),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
            "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-4, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-4, 10.0, log=True),
            "objective": "binary",
            "random_state": random_state,
            "verbose": -1,
        }

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="auc",
            callbacks=[lgb.log_evaluation(0)],
        )

        y_pred_proba = model.predict_proba(X_val)[:, 1]
        return roc_auc_score(y_val, y_pred_proba)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    best_params = study.best_params
    best_params["objective"] = "binary"
    best_params["random_state"] = random_state

    lgb_model = lgb.LGBMClassifier(**best_params)
    lgb_model.fit(X_train, y_train)

    y_val_proba = lgb_model.predict_proba(X_val)[:, 1]
    y_val_pred = lgb_model.predict(X_val)

    auc_score = roc_auc_score(y_val, y_val_proba)
    accuracy = accuracy_score(y_val, y_val_pred)

    print(f"Validation AUC: {auc_score:.6f}")
    print(f"Validation accuracy: {accuracy * 100:.2f}%")

    joblib.dump(lgb_model, os.path.join(model_dir, "model.pkl"))

    with open(os.path.join(model_dir, "best_params.json"), "w") as f:
        json.dump(best_params, f, indent=2)


### Prepare training inputs
Merge labels with the imputed feature matrix.

In [38]:
import pandas as pd
feature_matrix = pd.read_parquet("../data/intermediate/feature_matrix_imputed.parquet")
variant_labels = pd.read_csv("../data/intermediate/variant_labels.txt", sep="\t")
functional_training_input = variant_labels[["ID", "ensg", "functional_training_label"]].dropna(subset=["functional_training_label"]).merge(feature_matrix, how="left", on=["ID", "ensg"])
clinical_training_input = variant_labels[["ID", "ensg", "clinical_training_label"]].dropna(subset=["clinical_training_label"]).merge(feature_matrix, how="left", on=["ID", "ensg"])
del feature_matrix

### FuncVEP-CTI (clinical-trained predictors included)

In [ ]:
train_funcvep(functional_training_input, "FuncVEP_CTI", n_trials=50)

[I 2025-12-19 17:15:27,655] A new study created in memory with name: no-name-839918bc-25b7-4e6b-87b0-f44a6f2a2d89
[I 2025-12-19 17:15:28,923] Trial 0 finished with value: 0.9834684345021155 and parameters: {'num_leaves': 117, 'max_depth': 3, 'n_estimators': 481, 'subsample': 0.8542393487632973, 'subsample_freq': 3, 'colsample_bytree': 0.5006682198738345, 'learning_rate': 0.020529771765166023, 'min_child_samples': 159, 'min_child_weight': 0.31326361579003104, 'min_split_gain': 0.34887185544031596, 'lambda_l1': 0.00013571193257311707, 'lambda_l2': 0.04258362434663048}. Best is trial 0 with value: 0.9834684345021155.
[I 2025-12-19 17:15:30,637] Trial 1 finished with value: 0.98425390641364 and parameters: {'num_leaves': 66, 'max_depth': 9, 'n_estimators': 600, 'subsample': 0.8810596825551251, 'subsample_freq': 3, 'colsample_bytree': 0.3819622596474185, 'learning_rate': 0.03827828999315523, 'min_child_samples': 65, 'min_child_weight': 0.0011251013703794297, 'min_split_gain': 0.972728346013

Validation AUC: 0.985772
Validation accuracy: 94.51%


### FuncVEP-CTE (clinical-trained predictors excluded)

In [43]:
train_funcvep(functional_training_input, "FuncVEP_CTE", n_trials=50)

[I 2025-12-19 17:26:42,386] A new study created in memory with name: no-name-a00b73f7-5b62-43f0-b606-54418e296cd0
[I 2025-12-19 17:26:43,381] Trial 0 finished with value: 0.956332998198651 and parameters: {'num_leaves': 92, 'max_depth': 7, 'n_estimators': 167, 'subsample': 0.6670225632571246, 'subsample_freq': 2, 'colsample_bytree': 0.4257314632010271, 'learning_rate': 0.011427531471796811, 'min_child_samples': 115, 'min_child_weight': 1.9362573866900217, 'min_split_gain': 0.6648379802929562, 'lambda_l1': 0.748669103205949, 'lambda_l2': 0.0017341152671584848}. Best is trial 0 with value: 0.956332998198651.
[I 2025-12-19 17:26:44,637] Trial 1 finished with value: 0.9571551254660469 and parameters: {'num_leaves': 96, 'max_depth': 5, 'n_estimators': 158, 'subsample': 0.8398937057609318, 'subsample_freq': 1, 'colsample_bytree': 0.4382114842189193, 'learning_rate': 0.013957491758643012, 'min_child_samples': 54, 'min_child_weight': 0.174629383479943, 'min_split_gain': 0.4960969727112675, 'la

Validation AUC: 0.963716
Validation accuracy: 90.50%


### FuncVEP-SP (single predictor)

In [8]:
train_funcvep(functional_training_input, "FuncVEP_SP", n_trials=50)

[I 2025-12-19 15:49:50,050] A new study created in memory with name: no-name-5790f11f-678b-46af-bb7f-8486350d4731
[I 2025-12-19 15:49:51,662] Trial 0 finished with value: 0.9485830086716099 and parameters: {'num_leaves': 89, 'max_depth': 8, 'n_estimators': 525, 'subsample': 0.7099043915003347, 'subsample_freq': 1, 'colsample_bytree': 0.38670542613253267, 'learning_rate': 0.017686541860332042, 'min_child_samples': 99, 'min_child_weight': 6.339108840896202, 'min_split_gain': 0.5569123412556617, 'lambda_l1': 0.03587063354121592, 'lambda_l2': 0.41381403516801907}. Best is trial 0 with value: 0.9485830086716099.
[I 2025-12-19 15:49:53,825] Trial 1 finished with value: 0.9489181433538604 and parameters: {'num_leaves': 117, 'max_depth': 10, 'n_estimators': 555, 'subsample': 0.8964285536709179, 'subsample_freq': 5, 'colsample_bytree': 0.3020746123288997, 'learning_rate': 0.01654447072438871, 'min_child_samples': 84, 'min_child_weight': 0.011699180849918727, 'min_split_gain': 0.3551923202884007

Validation AUC: 0.950861
Validation accuracy: 88.33%


### ClinVEP-CTI (clinical-trained predictors included)

In [49]:
train_clinvep(clinical_training_input, "ClinVEP_CTI", n_trials=50)

[I 2025-12-19 17:59:56,910] A new study created in memory with name: no-name-49475fe2-432b-49a8-93fb-3ef61296e480
[I 2025-12-19 17:59:57,860] Trial 0 finished with value: 0.9957452099075825 and parameters: {'num_leaves': 114, 'max_depth': 7, 'n_estimators': 524, 'subsample': 0.9553313818874963, 'subsample_freq': 2, 'colsample_bytree': 0.7054511632147806, 'learning_rate': 0.06856976663949371, 'min_child_samples': 190, 'min_child_weight': 4.520047816064238, 'min_split_gain': 0.8063350197168115, 'lambda_l1': 0.10866606317751167, 'lambda_l2': 0.022279610277719224}. Best is trial 0 with value: 0.9957452099075825.
[I 2025-12-19 17:59:58,658] Trial 1 finished with value: 0.9919255688018899 and parameters: {'num_leaves': 100, 'max_depth': 3, 'n_estimators': 196, 'subsample': 0.8705890123233755, 'subsample_freq': 1, 'colsample_bytree': 0.6945442961050936, 'learning_rate': 0.012796586115460177, 'min_child_samples': 158, 'min_child_weight': 0.030381261269357904, 'min_split_gain': 0.73110332427021

Validation AUC: 0.996132
Validation accuracy: 97.50%


### ClinVEP_CTE (clinical-trained predictors excluded)

In [52]:
train_clinvep(clinical_training_input, "ClinVEP_CTE", n_trials=50)

[I 2025-12-19 18:09:32,916] A new study created in memory with name: no-name-8045e38b-8263-49e6-98e8-f30cff586261
[I 2025-12-19 18:09:33,952] Trial 0 finished with value: 0.9700990481979307 and parameters: {'num_leaves': 19, 'max_depth': 9, 'n_estimators': 123, 'subsample': 0.7783653827688948, 'subsample_freq': 2, 'colsample_bytree': 0.7598545424836491, 'learning_rate': 0.01935823556242671, 'min_child_samples': 44, 'min_child_weight': 0.11078356431093457, 'min_split_gain': 0.4133635378452146, 'lambda_l1': 0.0004579019925413218, 'lambda_l2': 0.27262703403249994}. Best is trial 0 with value: 0.9700990481979307.
[I 2025-12-19 18:09:35,635] Trial 1 finished with value: 0.9714114022848774 and parameters: {'num_leaves': 80, 'max_depth': 5, 'n_estimators': 388, 'subsample': 0.7124596837893066, 'subsample_freq': 3, 'colsample_bytree': 0.44030097197787893, 'learning_rate': 0.01071630981320862, 'min_child_samples': 97, 'min_child_weight': 0.003153355643992748, 'min_split_gain': 0.383664883871988

Validation AUC: 0.976067
Validation accuracy: 91.72%


### ClinVEP_SP (single predictor)

In [ ]:
train_clinvep(clinical_training_input, "ClinVEP_SP", n_trials=50)

[I 2025-12-19 18:13:06,460] A new study created in memory with name: no-name-61754a8c-ba6b-4bc7-968c-71ffc9027675
[I 2025-12-19 18:13:07,211] Trial 0 finished with value: 0.962991614748097 and parameters: {'num_leaves': 21, 'max_depth': 6, 'n_estimators': 118, 'subsample': 0.9663266683933113, 'subsample_freq': 5, 'colsample_bytree': 0.5920950792230368, 'learning_rate': 0.026099375258343004, 'min_child_samples': 60, 'min_child_weight': 0.02066959712128105, 'min_split_gain': 0.051595774513958026, 'lambda_l1': 1.2621953281220086, 'lambda_l2': 0.030924122892695822}. Best is trial 0 with value: 0.962991614748097.
[I 2025-12-19 18:13:09,660] Trial 1 finished with value: 0.9675434112917708 and parameters: {'num_leaves': 83, 'max_depth': 7, 'n_estimators': 544, 'subsample': 0.7260891822129909, 'subsample_freq': 2, 'colsample_bytree': 0.47768637581776496, 'learning_rate': 0.04082551111171829, 'min_child_samples': 191, 'min_child_weight': 0.013755910315495504, 'min_split_gain': 0.053803699938344

Validation AUC: 0.969857
Validation accuracy: 90.93%
